# Weighted-Mean Vertical Profiles and Initial Parameters by LCL Height Threshold

Purpose: MULCL-threshold counterpart to 11_weighted_mean_profiles_for_skewT.ipynb — compute
count-weighted mean temperature/dew point profiles and initial parcel parameters for the
two LCL-height-split subsets (>= 1500m and < 1500m) of lightning-only land points, as
preparation for Skew-T/log-P comparison between the two LCL threshold groups.

Input:
- Per-year pressure-level profile CSVs (MULCL-threshold subsets)
- Per-year screen-level CSVs
- Per-year initial parcel parameter CSVs (from initial_parcel_parameters_mulcl_threshold.py)

Output:
- Weighted-mean vertical profile CSVs per threshold group
- Combined weighted initial parameter summary for both groups

In [6]:
#this is calculate mean data parameter for cape plot with screen data, also simultaneously calculating 
#the weighted avergae of screen level
#weighted average of initial level parameters
# and weighted average at each pressure levels
import xarray as xr
import pandas as pd
import numpy as np
import os
import re

years = list(range(2016, 2025))
# folders = ['ls_only_land','ls_only_ocean','ls_ot_land','ls_ot_ocean','ot_only_land', 'ot_only_ocean'] #,'ls_ot_ocean'
folders = ['1_1_MULCL_less_1500m_mucape_0_ls_only_land_3d','1_1_MULCL_greater_equal_1500m_mucape_0_ls_only_land_3d']
barra_dir = '/g/data/if69/tp4064/Project_1/categorical_position/BARRA_data_skew_T/warm_Season_MULCL_condition'

rows = []
variables = ['MULPLdew','MULPLpres','MULPLtemp']


# screen_level_mean_MULCL_less_1500m_mucape_0_ls_only_land_3d.csv
# warm_Season_MULCL_condition/initial_param_MULCL_greater_equal_1500m_mucape_0_ls_only_land_3d.csv.csv")


for folder in folders:
    print(f"\n========== Processing folder: {folder} ==========")

    # --- Step 1: Sum 'count temp' across all years ---
    # looking over yearly pressure data for each categories
    file_list = glob.glob(f"{barra_dir}/{folder}_*.csv")
    count_df = None
    # print(file_list[0])

    for f in file_list:
        
        df = pd.read_csv(f, index_col=0)
        # print(df)
        col = pd.to_numeric(df['count_temp'], errors='coerce')
        if count_df is None:
            count_df = col.to_frame()
        else:
            count_df['count_temp'] += col
    
        
    # --- Step 2: Compute weighted mean ---
    temp = pd.DataFrame()
    dew  = pd.DataFrame()
    mean_data = pd.DataFrame()

    for year in years:
        
        file = glob.glob(f'{barra_dir}/{folder}_*{year}.csv')[0]

        
        
        data = pd.read_csv(file, usecols = ['pressure', 'temperature','dew_point','count_temp','count_dew'])
        # print(data)
        data.set_index(data.columns[0], inplace=True)
         # concating all year files 
        temp[year] = data['temperature'] * data['count_temp'] / count_df['count_temp']
        dew[year]  = data['dew_point']   * data['count_dew']  / count_df['count_temp'] #count_temp == count_dew

    # suming all those years along a pressure rows
        
    mean_data[f'{folder}_temperature'] = temp.sum(axis=1) - 273.15
    mean_data[f'{folder}_dew']         = dew.sum(axis=1)  - 273.15
    
    print(mean_data.reset_index(inplace = True))

    screen_file = glob.glob(f'{barra_dir}/1_screen_level*_{folder[2:]}.csv')[0]


    screen_data = pd.read_csv(screen_file)

    # screen_row = screen_data[screen_data['year'] == 'total']
    # print(screen_row)

    avg_temp = ((screen_data['surface_temp']*screen_data['count_temp'])/screen_data['count_temp'].sum()).sum()
    avg_dew = ((screen_data['surface_dew']*screen_data['count_dew'])/screen_data['count_dew'].sum()).sum()
    avg_press = ((screen_data['surface_press']*screen_data['Count_press'])/screen_data['Count_press'].sum()).sum()

    

    if  avg_press < 1000:
        
        mean_data.loc[mean_data['pressure'] == 1000,f'{folder}_temperature'] = (avg_temp -273.15)
        mean_data.loc[mean_data['pressure'] == 1000,f'{folder}_dew'] = (avg_dew-273.15)
        mean_data.loc[mean_data['pressure'] == 1000,'pressure'] = avg_press

    # print(mean_data)
    
    mean_data.to_csv(f"/g/data/if69/tp4064/Project_1/categorical_position/BARRA_data_skew_T/warm_Season_MULCL_condition/mean_data_{folder}.csv")


    # processing_initial_param

    initial_data = pd.read_csv(f'{barra_dir}/initial_param_{folder[2:]}.csv')

    df_pivot = initial_data.pivot(index = 'year', columns = 'Variable', values = ['mean','nonnan_count'])
    df_pivot.columns = ['_'.join(str(c) for c in col).strip('_') if isinstance(col, tuple) else col 
                         for col in df_pivot.columns]
    
    # # Find all variable names that have both mean_ and nonnan_count_ columns
    # variables = ['MULPLdew','MULPLpres','MULPLtemp']
    
    # Build the new row
    new_row = {col: None for col in df_pivot.columns}
    new_row['year'] = f'{folder[2:]}'
    
    for var in variables:
        mean_col = f'mean_{var}'
        count_col = f'nonnan_count_{var}'
        if mean_col in df_pivot.columns and count_col in df_pivot.columns:
            weighted_avg = (df_pivot[mean_col] * df_pivot[count_col]).sum() / df_pivot[count_col].sum()
            new_row[mean_col] = weighted_avg
    
    # Append it
    rows.append(new_row)
pd.DataFrame(rows).to_csv(f'/g/data/if69/tp4064/Project_1/categorical_position/BARRA_data_skew_T/warm_Season_MULCL_condition/mean_initial_param_MULCL_mucape_0_ls_only_land_3d.csv')


# print(f"\nSaved → {barra_dir}/mean_data_ot_only.csv")


========== Processing folder: 1_1_MULCL_less_1500m_mucape_0_ls_only_land_3d ==========
None

========== Processing folder: 1_1_MULCL_greater_equal_1500m_mucape_0_ls_only_land_3d ==========
None
